# Fine-Tuning the Generator

RAG handles factual grounding, but the generator still controls tone, format,
and answer structure. This notebook covers when fine-tuning the generator adds
value over RAG alone, and walks through a full LoRA fine-tuning run using `peft` and `trl`.

In [ ]:
# !pip install transformers peft trl bitsandbytes accelerate
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

## 0. Dependencies and Setup

Install all required libraries before running the notebook. The `datasets` library provides access to HuggingFace datasets. `trl` provides `SFTTrainer` and `SFTConfig`. `peft` provides LoRA. `bitsandbytes` enables 4-bit quantization (QLoRA).

In [ ]:
# !pip install transformers datasets peft trl bitsandbytes accelerate -q

import torch
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## When Fine-Tuning Helps

RAG provides the right facts. Fine-tuning teaches the model *how* to use them.
The two are complementary, not competing.

Fine-tuning the generator adds value when:

- **Output format is strict.** If answers must be JSON, or follow a citation template
  like `[ANSWER] ... [SOURCE: ...]`, prompting alone is fragile. Fine-tuning
  makes format compliance robust.

- **Domain tone matters.** Legal, medical, and customer-service QA each have
  register requirements that base models don't consistently maintain.

- **The base model ignores context.** Some smaller models are poor at conditioning
  on retrieved passages and default to parametric knowledge. A short SFT run
  on examples that demonstrate context usage usually fixes this.

Fine-tuning is *not* the fix for:

- **Stale facts.** The model's parametric knowledge doesn't update from SFT.
  If a model says the wrong year, retrieval is the answer, not fine-tuning.

- **Hallucination on topics outside the training set.** Fine-tuning on QA data
  teaches format and style, not factual accuracy on out-of-distribution queries.

## 1. Dataset Loading: Alpaca

The `tatsu-lab/alpaca` dataset contains 52k instruction-following examples in the style: instruction, optional input, output. It was originally used to fine-tune LLaMA and remains a standard benchmark for instruction tuning experiments.

Each row has three fields:
- `instruction`: the task description
- `input`: optional additional context (often empty)
- `output`: the desired model response

We load the first 2000 rows to keep training time under 10 minutes on a T4.

In [ ]:
from datasets import load_dataset

# Load first 2000 rows of the Alpaca dataset
alpaca_dataset = load_dataset("tatsu-lab/alpaca", split="train[:2000]")

print(f"Dataset size: {len(alpaca_dataset)} rows")
print(f"\nColumn names: {alpaca_dataset.column_names}")
print(f"\nDataset features:")
for col, feat in alpaca_dataset.features.items():
    print(f"  {col}: {feat}")

print("\n--- Sample row (index 0) ---")
sample = alpaca_dataset[0]
for key, val in sample.items():
    display_val = val[:200] + "..." if isinstance(val, str) and len(val) > 200 else val
    print(f"  {key}: {repr(display_val)}")

print("\n--- Sample row (index 42) ---")
sample2 = alpaca_dataset[42]
for key, val in sample2.items():
    display_val = val[:200] + "..." if isinstance(val, str) and len(val) > 200 else val
    print(f"  {key}: {repr(display_val)}")

# Show distribution of rows with vs. without an 'input' field
n_with_input = sum(1 for row in alpaca_dataset if row["input"].strip())
print(f"\nRows with a non-empty 'input' field: {n_with_input}/{len(alpaca_dataset)}")

## 2. Prompt Formatting with `apply_chat_template`

Modern instruction-tuned models use a chat format with system/user/assistant roles. `tokenizer.apply_chat_template` converts a list of message dicts into the model's expected string format, handling special tokens and delimiters correctly.

For Alpaca-style data:
- `instruction` (and optional `input`) become the user message.
- `output` becomes the assistant message appended after the generation prompt.

We add `<|endoftext|>` at the end so the model learns to stop generating after the answer.

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer early so we can use apply_chat_template for formatting
ALPACA_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
alpaca_tokenizer = AutoTokenizer.from_pretrained(ALPACA_MODEL_ID)
if alpaca_tokenizer.pad_token is None:
    alpaca_tokenizer.pad_token = alpaca_tokenizer.eos_token


def format_alpaca_row(row: dict, tokenizer) -> str:
    """
    Convert an Alpaca dataset row into a chat-formatted string for SFT.

    Args:
        row:       Dict with keys "instruction", "input", "output".
        tokenizer: HuggingFace tokenizer with apply_chat_template support.

    Returns:
        Fully formatted string ready for supervised fine-tuning.
        Includes the answer and an EOS token at the end.
    """
    # Combine instruction and input (if input is non-empty)
    user_content = row["instruction"]
    if row.get("input", "").strip():
        user_content += f"\n\n{row['input']}"

    messages = [
        {"role": "system",    "content": "You are a helpful assistant."},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": row["output"]},
    ]

    # add_generation_prompt=False because we include the full response
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    # Append EOS so the model learns to stop
    return formatted + tokenizer.eos_token


# --- Inspect a few formatted examples ---
for idx in [0, 1, 42]:
    row = alpaca_dataset[idx]
    formatted = format_alpaca_row(row, alpaca_tokenizer)
    print(f"=== Row {idx} ===")
    print(formatted[:600] + ("..." if len(formatted) > 600 else ""))
    token_count = len(alpaca_tokenizer.encode(formatted))
    print(f"[{token_count} tokens]\n")

# Check for very long rows that might exceed our max_seq_length
lengths = [len(alpaca_tokenizer.encode(format_alpaca_row(r, alpaca_tokenizer)))
           for r in alpaca_dataset]
import numpy as np
print(f"Token length stats: min={min(lengths)}, median={int(np.median(lengths))}, "
      f"p95={int(np.percentile(lengths, 95))}, max={max(lengths)}")

## 3. QLoRA Configuration: `BitsAndBytesConfig` + `LoraConfig`

QLoRA (Quantized LoRA) combines two techniques:

1. **4-bit NF4 quantization (`BitsAndBytesConfig`):** The base model weights are stored in 4-bit NF4 format, cutting memory by ~4x compared to BF16. Computation still happens in BF16 for numerical stability.

2. **LoRA adapter (`LoraConfig`):** Small rank-decomposition matrices are added to selected attention layers. Only these matrices are trained; the frozen base weights are never updated.

Key parameters for `LoraConfig`:
- `r=16`: rank of the adapter matrices. Higher rank = more capacity, more parameters.
- `lora_alpha=32`: scaling factor. Standard heuristic: `lora_alpha = 2 * r`.
- `target_modules`: which weight matrices to adapt. Including all four attention projections (`q_proj`, `v_proj`, `k_proj`, `o_proj`) gives better results than just q+v, especially for instruction-following.
- `lora_dropout=0.05`: regularization to prevent overfitting on small datasets.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# --- Memory checkpoint 1: before model load ---
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    mem_before_load = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM before model load: {mem_before_load:.3f} GB")

# 4-bit NF4 quantization config
qlora_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,    # saves ~0.4 GB more via double quantization
)

print(f"\nLoading {ALPACA_MODEL_ID} with QLoRA config ...")
qlora_model = AutoModelForCausalLM.from_pretrained(
    ALPACA_MODEL_ID,
    quantization_config=qlora_bnb_config,
    device_map="auto",
)
qlora_model.config.use_cache = False          # needed for gradient checkpointing

# --- Memory checkpoint 2: after base model load ---
if torch.cuda.is_available():
    mem_after_load = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM after base model load: {mem_after_load:.3f} GB  "
          f"(delta: {mem_after_load - mem_before_load:.3f} GB)")

# LoRA configuration with all four attention projections
qlora_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,                       # 2 * r is a good starting point
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

qlora_model = get_peft_model(qlora_model, qlora_lora_config)
print("\nLoRA adapters attached. Configuration summary:")
print(f"  r:               {qlora_lora_config.r}")
print(f"  lora_alpha:      {qlora_lora_config.lora_alpha}")
print(f"  target_modules:  {qlora_lora_config.target_modules}")
print(f"  lora_dropout:    {qlora_lora_config.lora_dropout}")

## 3a. Trainable Parameter Count

One of LoRA's main advantages is that it adds very few trainable parameters. We can verify this precisely and see exactly which modules have trainable weights.

In [ ]:
def print_trainable_parameters(model) -> None:
    """
    Print a summary of trainable vs. total parameters in the model.
    Also lists the first 10 trainable parameter names.
    """
    trainable_params = 0
    total_params = 0
    trainable_names = []

    for name, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
            trainable_names.append(name)

    pct = 100 * trainable_params / total_params if total_params > 0 else 0

    print("=" * 60)
    print("Trainable Parameter Summary")
    print("=" * 60)
    print(f"  Total parameters:     {total_params:>15,}")
    print(f"  Trainable parameters: {trainable_params:>15,}")
    print(f"  Frozen parameters:    {total_params - trainable_params:>15,}")
    print(f"  Trainable fraction:   {pct:>14.4f}%")
    print()
    print(f"First 10 trainable parameter tensors:")
    for name in trainable_names[:10]:
        param = dict(model.named_parameters())[name]
        print(f"  {name:<55} shape={list(param.shape)}")
    if len(trainable_names) > 10:
        print(f"  ... and {len(trainable_names) - 10} more")


print_trainable_parameters(qlora_model)

## 4. Preparing the Alpaca Dataset for SFTTrainer

`SFTTrainer` expects a dataset with a `text` column containing fully-formatted strings. We apply `format_alpaca_row` to every row and also truncate to `max_seq_length=512` to prevent memory spikes on rare long examples.

In [ ]:
from datasets import Dataset as HFDataset

MAX_SEQ_LENGTH = 512

def prepare_alpaca_sft(dataset, tokenizer, max_seq_length: int = 512) -> HFDataset:
    """
    Format an Alpaca-style HuggingFace dataset for SFTTrainer.
    Rows that exceed max_seq_length are discarded.
    """
    formatted_rows = []
    skipped = 0
    for row in dataset:
        text = format_alpaca_row(row, tokenizer)
        token_len = len(tokenizer.encode(text))
        if token_len <= max_seq_length:
            formatted_rows.append({"text": text})
        else:
            skipped += 1
    print(f"Formatted {len(formatted_rows)} rows, skipped {skipped} rows "
          f"(>{max_seq_length} tokens).")
    return HFDataset.from_list(formatted_rows)


sft_dataset = prepare_alpaca_sft(alpaca_dataset, alpaca_tokenizer, MAX_SEQ_LENGTH)

print(f"\nSFT dataset size: {len(sft_dataset)}")
print("\nFirst formatted example (truncated):")
print(sft_dataset[0]["text"][:500] + "...")

## 5. SFTConfig + SFTTrainer

`SFTTrainer` from `trl` is a thin wrapper around HuggingFace `Trainer` with sensible defaults for supervised fine-tuning:

- `packing=True`: packs multiple short examples into a single sequence up to `max_seq_length`. This increases GPU utilization significantly on datasets with many short examples.
- `max_seq_length=512`: clips any sequence beyond this length.
- `gradient_accumulation_steps=4`: simulates a larger effective batch size (4 * 2 = 8) without needing more VRAM.
- `bf16=True`: use BF16 for gradient accumulation (requires Ampere GPU or newer; fall back to `fp16=True` on T4).

A custom `TrainerCallback` collects loss values at each logging step so we can plot the training curve after the run completes.

In [ ]:
import torch
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


# --- Custom callback to collect training loss ---
class LossCollectorCallback(TrainerCallback):
    """Accumulate (step, loss) pairs during training for post-hoc plotting."""

    def __init__(self):
        self.steps = []
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.steps.append(state.global_step)
            self.losses.append(logs["loss"])


loss_callback = LossCollectorCallback()

# --- VRAM before training ---
if torch.cuda.is_available():
    mem_before_train = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM before training starts: {mem_before_train:.3f} GB")

# SFTConfig: training hyperparameters
sft_config = SFTConfig(
    output_dir="/tmp/alpaca_qlora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,        # effective batch size = 8
    learning_rate=2e-4,
    fp16=False,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    logging_steps=10,
    save_strategy="no",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,                          # pack short examples into full-length sequences
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=True,           # trade compute for memory
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

# SFTTrainer
sft_trainer = SFTTrainer(
    model=qlora_model,
    train_dataset=sft_dataset,
    args=sft_config,
    tokenizer=alpaca_tokenizer,
    callbacks=[loss_callback],
)

print(f"\nStarting training on {len(sft_dataset)} examples ...")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size (per device): {sft_config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Max seq length: {sft_config.max_seq_length}")
print(f"  Packing: {sft_config.packing}\n")

train_result = sft_trainer.train()

# --- VRAM during/after training ---
if torch.cuda.is_available():
    mem_peak_train = torch.cuda.max_memory_allocated() / 1e9
    mem_after_train = torch.cuda.memory_allocated() / 1e9
    print(f"\nVRAM peak during training: {mem_peak_train:.3f} GB")
    print(f"VRAM after training:       {mem_after_train:.3f} GB")

print(f"\nTraining complete. Final loss: {train_result.training_loss:.4f}")

## 5a. Training Loss Plot

Plot the loss curve collected by `LossCollectorCallback`. A healthy curve should show a steady decline that flattens as the model converges. If the curve is erratic or does not decrease, check the learning rate and effective batch size.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not loss_callback.steps:
    print("No loss data collected. Make sure training ran before this cell.")
else:
    steps  = loss_callback.steps
    losses = loss_callback.losses

    # Smooth the curve with a rolling average for readability
    window = max(1, len(losses) // 10)
    smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
    smoothed_steps = steps[window - 1:]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, losses, color="lightsteelblue", linewidth=1, alpha=0.6, label="Raw loss")
    ax.plot(smoothed_steps, smoothed, color="steelblue", linewidth=2, label=f"Smoothed (window={window})")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Curve (QLoRA on Alpaca-2k)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)

    print(f"Steps logged: {len(steps)}")
    print(f"Initial loss: {losses[0]:.4f}")
    print(f"Final loss:   {losses[-1]:.4f}")
    print(f"Reduction:    {losses[0] - losses[-1]:.4f}  ({(losses[0] - losses[-1]) / losses[0] * 100:.1f}%)")

    plt.tight_layout()
    plt.show()

## 6. Merge and Save the Fine-Tuned Model

After training, the model holds a quantized base + LoRA adapters. For deployment you have two options:

1. **Adapter only (small):** Save just the adapter weights (~20-50 MB). Load with `PeftModel.from_pretrained` at inference time. Good for iterating quickly.
2. **Merged model (larger):** Call `model.merge_and_unload()` to fold the adapter weights back into the base weights, then save the result as a standard HuggingFace model. Good for serving -- no PEFT dependency at runtime.

We do both below.

In [ ]:
import os

ADAPTER_DIR = "/tmp/alpaca_qlora_adapter"
MERGED_DIR  = "/tmp/alpaca_qlora_merged"

# --- Option 1: Save just the LoRA adapter weights ---
qlora_model.save_pretrained(ADAPTER_DIR)
alpaca_tokenizer.save_pretrained(ADAPTER_DIR)

adapter_files = os.listdir(ADAPTER_DIR)
adapter_size  = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
                    for f in adapter_files if os.path.isfile(os.path.join(ADAPTER_DIR, f)))
print(f"Adapter saved to {ADAPTER_DIR}")
print(f"  Files: {adapter_files}")
print(f"  Total size: {adapter_size / 1e6:.1f} MB\n")

# --- Option 2: Merge LoRA into base weights and save ---
# merge_and_unload() dequantizes, merges, and returns a plain nn.Module
print("Merging LoRA adapter into base model weights ...")
merged_model = qlora_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR)
alpaca_tokenizer.save_pretrained(MERGED_DIR)

merged_files = os.listdir(MERGED_DIR)
merged_size  = sum(os.path.getsize(os.path.join(MERGED_DIR, f))
                   for f in merged_files if os.path.isfile(os.path.join(MERGED_DIR, f)))
print(f"Merged model saved to {MERGED_DIR}")
print(f"  Files: {sorted(merged_files)}")
print(f"  Total size: {merged_size / 1e6:.1f} MB")
print("\nThe merged model can be loaded with AutoModelForCausalLM.from_pretrained(MERGED_DIR)")
print("without any PEFT dependency.")

## 7. Before vs. After: Response Comparison

The most direct way to assess fine-tuning impact is to run the same prompts through the base model and the fine-tuned model and compare their outputs side by side.

We use the existing `code-before` `generate_answer` function but now generate from three held-out prompts covering different instruction types.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Load the base model (no LoRA) for comparison
base_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading base model {ALPACA_MODEL_ID} for comparison ...")
base_model_cmp = AutoModelForCausalLM.from_pretrained(
    ALPACA_MODEL_ID,
    quantization_config=base_bnb_config,
    device_map="auto",
)
cmp_tokenizer = AutoTokenizer.from_pretrained(ALPACA_MODEL_ID)

# Load the fine-tuned (merged) model
print(f"Loading fine-tuned model from {MERGED_DIR} ...")
ft_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Comparison prompts (instruction-following tasks from outside the training set)
comparison_prompts = [
    "Give three tips for writing clean Python code.",
    "Explain the difference between a stack and a queue in one paragraph.",
    "Write a short poem about the color blue.",
]


def generate_from_instruction(model, tokenizer, instruction: str, max_new_tokens: int = 200) -> str:
    """Generate a response for a plain instruction using the chat template."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


# Run all three prompts through both models
for i, prompt in enumerate(comparison_prompts, 1):
    base_response = generate_from_instruction(base_model_cmp, cmp_tokenizer, prompt)
    ft_response   = generate_from_instruction(ft_model, cmp_tokenizer, prompt)

    print(f"{'='*65}")
    print(f"Prompt {i}: {prompt}")
    print(f"\n[BASE MODEL]")
    print(base_response)
    print(f"\n[FINE-TUNED MODEL]")
    print(ft_response)
    print()

print("Observations to look for:")
print("- Does the fine-tuned model follow the instruction format more consistently?")
print("- Is the tone/style different after training on Alpaca?")
print("- Are there any regressions (tasks where the base model did better)?")

## Instruction Tuning vs. Task-Specific Tuning

**Instruction tuning** trains the model on a diverse set of (instruction, response) pairs.
The goal is a model that generalizes: follow any instruction, answer any question.
Most chat models (Mistral-Instruct, Llama-3-Instruct, Phi-3-mini) have already been
through this stage.

**Task-specific tuning** fine-tunes on a narrow distribution that matches your deployment.
You trade generality for reliability on that specific task.

For RAG QA, the typical prompt template looks like this:

In [ ]:
def format_rag_prompt(context: str, question: str, answer: str = None) -> str:
    """
    Format a QA example for supervised fine-tuning.
    If answer is None, returns the prompt-only version (for inference).
    """
    prompt = (
        "<|system|>\n"
        "You are a helpful assistant that answers questions using only "
        "the provided context. If the context does not contain the answer, "
        "say so clearly.\n"
        "<|user|>\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "<|assistant|>\n"
    )
    if answer is not None:
        prompt += answer + "<|endoftext|>"
    return prompt


example = format_rag_prompt(
    context="The Eiffel Tower was completed in 1889 and stands 330 meters tall.",
    question="How tall is the Eiffel Tower?",
    answer="The Eiffel Tower stands 330 meters tall.",
)
print(example)

## LoRA Configuration

LoRA (Low-Rank Adaptation) freezes the original weights and adds small trainable
rank-decomposition matrices to selected layers. For a 1.5B parameter model,
a rank-8 LoRA on the attention projections adds roughly 3-5M trainable parameters,
compared to 1.5B for full fine-tuning.

Key parameters:
- `r`: the rank. Higher rank = more capacity = more memory. 8 or 16 is typical for QA.
- `lora_alpha`: scaling factor. A common heuristic is `lora_alpha = 2 * r`.
- `target_modules`: which weight matrices to adapt. Query and value projections
  are the standard choice; adding `k_proj` and `o_proj` helps with longer contexts.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization to reduce memory during training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # required for gradient checkpointing

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Preparing the Training Data

SFTTrainer from `trl` expects a dataset with a `text` column containing the
fully formatted prompt (including the answer for training examples).
The trainer handles tokenization and packing internally.

In [ ]:
# Synthetic QA dataset: (context, question, answer)
raw_data = [
    ("Mount Everest is 8,849 meters above sea level and is located in the Himalayas.",
     "What is the height of Mount Everest?",
     "Mount Everest is 8,849 meters above sea level."),
    ("Python was created by Guido van Rossum and first released in 1991.",
     "Who created the Python programming language?",
     "Python was created by Guido van Rossum."),
    ("The speed of light in a vacuum is approximately 299,792 kilometers per second.",
     "How fast does light travel in a vacuum?",
     "Light travels at approximately 299,792 kilometers per second in a vacuum."),
    ("The Amazon River is the largest river by discharge volume in the world, located in South America.",
     "Where is the Amazon River located?",
     "The Amazon River is located in South America."),
    ("DNA stands for deoxyribonucleic acid and carries genetic information in living organisms.",
     "What does DNA stand for?",
     "DNA stands for deoxyribonucleic acid."),
    ("The Great Wall of China was built over many centuries, primarily during the Ming dynasty.",
     "During which dynasty was most of the Great Wall of China built?",
     "Most of the Great Wall was built during the Ming dynasty."),
    ("Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose.",
     "What inputs do plants use in photosynthesis?",
     "Plants use sunlight, water, and CO2 as inputs for photosynthesis."),
    ("The human genome contains approximately 3 billion base pairs and around 20,000 genes.",
     "How many genes does the human genome contain?",
     "The human genome contains around 20,000 genes."),
    ("Jupiter is the largest planet in the solar system with a mass greater than all other planets combined.",
     "What is the largest planet in our solar system?",
     "Jupiter is the largest planet in the solar system."),
    ("The French Revolution began in 1789 and led to the overthrow of the French monarchy.",
     "What did the French Revolution lead to?",
     "The French Revolution led to the overthrow of the French monarchy."),
    ("Antibiotics are medications that kill or inhibit the growth of bacteria. They do not work on viruses.",
     "Do antibiotics work against viral infections?",
     "No, antibiotics do not work on viruses. They only target bacteria."),
    ("The context does not contain information about the capital of Australia.",
     "What is the capital of Australia?",
     "The provided context does not contain information about the capital of Australia."),
]

formatted = [{"text": format_rag_prompt(ctx, q, a)} for ctx, q, a in raw_data]
dataset = Dataset.from_list(formatted)

print(f"Dataset size: {len(dataset)}")
print("\nSample entry:")
print(dataset[0]["text"])

## Baseline: Inference Before Fine-Tuning

Run the held-out questions through the base model before training.
Save the outputs to compare against the post-training results.

In [ ]:
held_out = [
    (
        "The Nile is the longest river in Africa, stretching approximately 6,650 km.",
        "How long is the Nile River?",
    ),
    (
        "This context does not mention anything about the boiling point of water.",
        "At what temperature does water boil?",
    ),
]


def generate_answer(model, tokenizer, context, question, max_new_tokens=128):
    prompt = format_rag_prompt(context, question)  # no answer = inference mode
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("=== Before Fine-Tuning ===")
for ctx, q in held_out:
    answer = generate_answer(model, tokenizer, ctx, q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

In [ ]:
training_args = SFTConfig(
    output_dir="/tmp/rag_generator_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
)

trainer.train()
print("Training complete.")

In [ ]:
print("=== After Fine-Tuning ===")
for ctx, q in held_out:
    answer = generate_answer(model, tokenizer, ctx, q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

# What to look for:
# 1. For the Nile question: the model should answer using the context,
#    not drift to parametric knowledge.
# 2. For the boiling point question: the model should acknowledge the
#    context does not contain the answer, rather than hallucinating.

## Saving and Merging the LoRA Adapter

LoRA training produces adapter weights, not a full model. You can save just the adapter
(small, ~20MB) or merge it back into the base model for deployment.

In [ ]:
# Save only the adapter weights
model.save_pretrained("/tmp/rag_lora_adapter")
tokenizer.save_pretrained("/tmp/rag_lora_adapter")
print("Adapter saved.")

# To deploy, merge the adapter into the base model:
# merged = model.merge_and_unload()
# merged.save_pretrained("/tmp/rag_merged_model")